# ⏱️ LOS Analysis - Length of Stay Prediction
## تحليل شامل لمدة الإقامة في المستشفى
---

In [ ]:
# ================== استيراد المكتبات ==================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# إعدادات الرسوم
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ المكتبات جاهزة")

In [ ]:
# ================== تحميل البيانات ==================
base = '/content/drive2/MyDrive/RIVA-Maternal/los/data'

# تحميل الميزات
X = np.load(f'{base}/X_features.npy', allow_pickle=True)
y = np.load(f'{base}/y_target.npy', allow_pickle=True)

with open(f'{base}/feature_names.json', 'r') as f:
    feature_info = json.load(f)
    feature_names = feature_info['features']
    los_99 = feature_info.get('los_99', 30.47)

df = pd.DataFrame(X, columns=feature_names)
df['los_days'] = y

print(f"✅ تم تحميل {len(df)} عينة")
print(f"📊 عدد الميزات: {len(feature_names)}")
print(df.head())

## 📊 تحليل إحصائي أولي

In [ ]:
# إحصائيات LOS
print("📈 إحصائيات LOS:")
print(df['los_days'].describe())

# رسم توزيع LOS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['los_days'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('توزيع LOS')
axes[0].set_xlabel('أيام')
axes[0].set_ylabel('عدد')

axes[1].boxplot(df['los_days'])
axes[1].set_title('صندوق LOS')
axes[1].set_ylabel('أيام')

plt.tight_layout()
plt.show()

## 🔥 أهمية الميزات

In [ ]:
# تحميل النموذج المحسن
model_path = '/content/drive2/MyDrive/RIVA-Maternal/los/models/xgb_los_improved.pkl'
model = joblib.load(model_path)

# أهمية الميزات
importance = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('الأهمية')
plt.title('Feature Importance - LOS')
plt.gca().invert_yaxis()
plt.show()

print(importance)

## 🎯 تقييم النموذج

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# توقع
y_pred = model.predict(X_test)

# مقاييس
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"📊 نتائج التقييم:")
print(f"   MAE: {mae:.3f} يوم")
print(f"   RMSE: {rmse:.3f} يوم")
print(f"   R²: {r2:.3f}")

# رسم Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([0, los_99], [0, los_99], 'r--', linewidth=2)
plt.xlabel('القيمة الفعلية (أيام)')
plt.ylabel('القيمة المتوقعة (أيام)')
plt.title(f'LOS: Actual vs Predicted (MAE={mae:.2f})')
plt.tight_layout()
plt.show()

## 📈 تحليل الأخطاء

In [ ]:
errors = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(errors, bins=20, edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('خطأ التوقع (أيام)')
axes[0].set_ylabel('عدد')
axes[0].set_title('توزيع الأخطاء')

axes[1].boxplot(errors)
axes[1].set_title('صندوق الأخطاء')
axes[1].set_ylabel('أيام')

plt.tight_layout()
plt.show()

print(f"📊 إحصائيات الأخطاء:")
print(f"   المتوسط: {errors.mean():.3f}")
print(f"   الانحراف: {errors.std():.3f}")
print(f"   المدى: [{errors.min():.2f}, {errors.max():.2f}]")

## ✅ الخلاصة

In [ ]:
print("\n" + "="*60)
print("📊 ملخص تحليل LOS")
print("="*60)
print(f"""
🎯 نتائج النموذج المحسن:
   - MAE: {mae:.3f} يوم (متوسط الخطأ)
   - RMSE: {rmse:.3f} يوم
   - R²: {r2:.3f}

🔥 أهم الميزات:
{importance.to_string(index=False)}

📈 توصيات:
1. الميزات السريرية (الأدوية، الإجراءات) هي الأهم
2. النموذج الحالي يعمل بكفاءة على عينة {len(df)} مريض
3. مع بيانات أكبر، الأداء سيكون أفضل
""")